# 02 - Treinamento e avaliação dos modelos

## Ordem de execução

As hipóteses são avaliadas nesta ordem, e a ordem não é negociável:

1. **H1** - desempenho preditivo, no conjunto de **validação**
2. **H2** - custo computacional e complexidade, no conjunto de **validação**
3. **Congelamento** dos resultados
4. **H3** - generalização, no conjunto **independente (Hannousse)**, uma única vez


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from avaliacao import avaliar_completo, avaliar_predicao, ordens_de_magnitude

PROC = Path("../dados/processados")
RESULTADOS = Path("../resultados")
TABELAS = RESULTADOS / "tabelas"
TABELAS.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Carregamento dos dados

Os seis arquivos estão na pasta de dados processados e foram gerados pelo mesmo módulo de extração. Ele calcula os 37 atributos lexicais a partir das URLs brutas, sem usar nenhum atributo que já viesse pronto nas bases originais. O treino (188.296 URLs) e a validação (47.074) vêm do PhiUSIIL. Depois de retirar as URLs repetidas, a base foi dividida em 80% e 20%, com estratificação pelo rótulo e random_state = 42, e por isso as duas partições têm a mesma proporção de phishing (42,7%). O teste independente (11.385 URLs) vem do conjunto de Hannousse e Yahiouche, coletado por outros autores, em outro momento e com outras fontes. Dele foram retiradas as URLs repetidas e as que também apareciam no PhiUSIIL. Essa partição não participou de nenhuma decisão de projeto e só foi usada depois que os modelos estavam fechados. É por isso que ela mede se o que foi aprendido se transfere para dados de outra origem.

In [2]:
def carregar(nome):
    return pd.read_csv(PROC / f"{nome}.csv", index_col=0)

X_tr    = carregar("X_tr")
X_val   = carregar("X_val")
X_teste = carregar("X_teste")

y_tr    = carregar("y_tr")["y"]
y_val   = carregar("y_val")["y"]
y_teste = carregar("y_teste")["y"]

print(f"treino     {X_tr.shape}   phishing: {y_tr.mean():.3f}")
print(f"validação  {X_val.shape}   phishing: {y_val.mean():.3f}")
print(f"teste ind. {X_teste.shape}   phishing: {y_teste.mean():.3f}")

treino     (188296, 37)   phishing: 0.427
validação  (47074, 37)   phishing: 0.427
teste ind. (11385, 37)   phishing: 0.498


In [3]:
# Verificações que devem passar antes de qualquer treino.
assert list(X_tr.columns) == list(X_val.columns) == list(X_teste.columns), \
    "As três partições precisam ter os mesmos atributos na mesma ordem."
assert not X_tr.isna().any().any(), "Há valores ausentes no treino."
assert not X_val.isna().any().any(), "Há valores ausentes na validação."
assert not X_teste.isna().any().any(), "Há valores ausentes no teste independente."
assert set(y_tr.unique()) <= {0, 1}, "Rótulos fora da convenção binária."

print(f"{X_tr.shape[1]} atributos, verificações aprovadas.")

37 atributos, verificações aprovadas.


## 2. Definição dos modelos

Os quatro algoritmos foram escolhidos para cobrir uma escala de complexidade, do mais simples ao mais elaborado. A Regressão Logística é a referência linear e mostra até onde se chega separando as classes com uma fronteira simples. A Árvore de Decisão capta interações entre atributos e continua fácil de interpretar. O Random Forest e o XGBoost são ensembles de árvores construídos de formas diferentes: o primeiro combina árvores independentes (bagging), e o segundo corrige os erros da árvore anterior a cada etapa (boosting). Essa escala permite testar se o ganho dos modelos mais complexos compensa o custo que eles têm. O SVM com kernel RBF ficou de fora por um motivo prático. Segundo a documentação do scikit-learn, o tempo de ajuste cresce pelo menos de forma quadrática com o número de amostras, e com quase 190 mil URLs no treino isso seria inviável. A exclusão diz respeito à viabilidade, não ao desempenho do modelo.

O desbalanceamento é tratado exclusivamente por ponderação de classes durante
o treino. Nenhuma reamostragem é aplicada, e nenhuma das partições de
avaliação é rebalanceada.

In [4]:
# Proporção usada na ponderação do XGBoost (negativos / positivos).
peso_positivo = float((y_tr == 0).sum() / (y_tr == 1).sum())
print(f"scale_pos_weight = {peso_positivo:.4f}")

modelos = {
    "Regressão Logística": Pipeline([
        ("escala", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=SEMENTE,
        )),
    ]),
    "Árvore de Decisão": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=SEMENTE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=1,          # fixado para que a medição de tempo seja comparável
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        scale_pos_weight=peso_positivo,
        random_state=SEMENTE,
        n_jobs=1,
        tree_method="hist",
        eval_metric="logloss",
    ),
}

for nome in modelos:
    print(nome)

scale_pos_weight = 1.3415
Regressão Logística
Árvore de Decisão
Random Forest
XGBoost


### Nota sobre a padronização

A Regressão Logística é o único modelo em Pipeline com `StandardScaler`.
Modelos baseados em árvore particionam por limiar em cada atributo
isoladamente e são invariantes a transformações monotônicas de escala;
padronizá-los não mudaria as partições e apenas acrescentaria custo.

O `StandardScaler` é ajustado dentro do Pipeline, portanto suas estatísticas
vêm apenas do conjunto de treino. Ajustá-lo sobre os dados completos
vazaria informação das partições de avaliação.

## 3. Treinamento

 Os quatro modelos foram treinados uma única vez, no mesmo computador, e o tempo medido inclui só a chamada a fit; a extração dos atributos fica de fora. A Árvore de Decisão foi a mais rápida (1,07 s), seguida pela Regressão Logística (1,94 s) e pelo XGBoost (2,30 s). O Random Forest levou bem mais, 15,59 s, porque constrói 100 árvores profundas e independentes, enquanto o XGBoost usa árvores rasas. Como foi uma só medição, esses valores servem para comparar a ordem de grandeza entre os modelos, não como tempos exatos. Outra máquina, ou outra execução, pode dar números um pouco diferentes.

In [5]:
tempos_treino = {}

for nome, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    tempos_treino[nome] = time.perf_counter() - inicio
    print(f"{nome:22s} {tempos_treino[nome]:7.2f}s")

Regressão Logística       1.94s


Árvore de Decisão         1.07s


Random Forest            15.59s


XGBoost                   2.30s


## 4. H1 - desempenho preditivo

**Hipótese:** os modelos de ensemble superam os modelos de base em F1.

A avaliação é feita no conjunto de validação, extraído do PhiUSIIL. A classe
positiva é phishing.

Na validação, os quatro modelos tiveram um desempenho quase perfeito, com F1 entre 0,9951 e 0,9970. Os dois ensembles ficaram na frente: XGBoost com 0,9970 e Random Forest com 0,9969, contra 0,9957 da Árvore de Decisão e 0,9951 da Regressão Logística. Pelo critério definido antes do experimento, H1 está corroborada. A diferença, porém, é muito pequena. O XGBoost erra 122 das 47.074 URLs e a Árvore de Decisão, 172: são 50 erros a menos em mais de 47 mil casos. O XGBoost é o modelo com menos falsos positivos (12). A Regressão Logística é o que mais deixa passar phishing (173 falsos negativos). Valores tão altos medidos na mesma base usada no treino ainda não dizem se os modelos aprenderam o fenômeno ou só as características dessa coleta. Essa resposta vem do teste independente.

In [6]:
registros = [
    avaliar_completo(nome, modelo, X_val, y_val)
    for nome, modelo in modelos.items()
]

res_val = pd.DataFrame(registros).set_index("modelo")
res_val["tempo_treino_s"] = pd.Series(tempos_treino)

colunas_h1 = ["f1", "precisao", "recall", "roc_auc", "pr_auc", "vp", "fp", "fn"]
res_val[colunas_h1].round(4)

,f1,precisao,recall,roc_auc,pr_auc,vp,fp,fn
modelo,,,,,,,,
Regressão Logística,0.9951,0.9989,0.9914,0.9980,0.9986,19931,22,173
Árvore de Decisão,0.9957,0.9964,0.9950,0.9963,0.9943,20004,72,100
Random Forest,0.9969,0.9989,0.9949,0.9984,0.9984,20002,22,102
XGBoost,0.9970,0.9994,0.9945,0.9983,0.9988,19994,12,110


In [7]:
# Verificação formal da H1.
BASE = ["Regressão Logística", "Árvore de Decisão"]
ENSEMBLE = ["Random Forest", "XGBoost"]

f1_base_max = res_val.loc[BASE, "f1"].max()
f1_ens_max = res_val.loc[ENSEMBLE, "f1"].max()

print(f"melhor F1 entre os modelos de base:      {f1_base_max:.4f}")
print(f"melhor F1 entre os modelos de ensemble:  {f1_ens_max:.4f}")
print(f"ganho absoluto:                          {f1_ens_max - f1_base_max:+.4f}")
print()
print("H1 corroborada." if f1_ens_max > f1_base_max else "H1 NÃO corroborada.")

melhor F1 entre os modelos de base:      0.9957
melhor F1 entre os modelos de ensemble:  0.9970
ganho absoluto:                          +0.0012

H1 corroborada.


## 5. H2 - custo computacional e complexidade

**Hipótese:** o ganho de F1 dos ensembles é inferior a 0,05 em valor absoluto,
enquanto o tempo de inferência e o número de nós aumentam em pelo menos uma
ordem de magnitude.

O limiar de 0,05 e o critério de uma ordem de magnitude foram fixados antes da
observação dos resultados e não são revisáveis.

O ganho de desempenho dos ensembles foi muito pequeno. O XGBoost, o melhor deles, ficou 0,0012 ponto de F1 acima da Árvore de Decisão, bem abaixo do limite de 0,05, e a primeira condição de H2 foi atendida. O aumento de custo, porém, não chegou a uma ordem de grandeza. O XGBoost levou 1,58 µs por URL, contra 0,20 µs da árvore: é cerca de oito vezes mais lento, o equivalente a 0,9 ordem de grandeza. O número de nós foi só 1,6 vez maior (4.156 contra 2.649), porque o XGBoost usa árvores rasas, com profundidade 6. Pelo critério definido antes do experimento, H2 foi refutada.

O Random Forest mostra o outro lado: ele teria cumprido as duas condições de custo, com tempo cerca de 51 vezes maior e quase 105 vezes mais nós, mas não é o ensemble de melhor F1. A comparação mostra que um modelo mais complexo nem sempre custa muito mais. Também mostra que pagar mais, como no caso do Random Forest, não trouxe ganho que justificasse o custo. Neste conjunto, a Árvore de Decisão oferece quase o mesmo resultado pelo menor preço.

In [8]:
colunas_h2 = [
    "f1", "tempo_mediana_us", "tempo_desvio_us",
    "n_arvores", "n_nos_total", "profundidade_max", "tipo_complexidade",
]
res_val[colunas_h2].round(4)

,f1,tempo_mediana_us,tempo_desvio_us,n_arvores,n_nos_total,profundidade_max,tipo_complexidade
modelo,,,,,,,
Regressão Logística,0.9951,0.3384,0.0409,0,37,0,coeficientes
Árvore de Decisão,0.9957,0.1990,0.0037,1,2649,51,nos_arvore
Random Forest,0.9969,10.1719,0.0665,100,277014,49,nos_arvore
XGBoost,0.9970,1.5841,0.0109,100,4156,6,nos_arvore


In [9]:
# Verificação formal da H2, usando a Árvore de Decisão como referência
# de complexidade: é o modelo de base que possui nós, o que torna a
# comparação de número de nós uma comparação entre grandezas equivalentes.
referencia = "Árvore de Decisão"
melhor_ensemble = res_val.loc[ENSEMBLE, "f1"].idxmax()

ganho_f1 = res_val.loc[melhor_ensemble, "f1"] - f1_base_max
om_tempo = ordens_de_magnitude(
    res_val.loc[melhor_ensemble, "tempo_mediana_us"],
    res_val.loc[referencia, "tempo_mediana_us"],
)
om_nos = ordens_de_magnitude(
    res_val.loc[melhor_ensemble, "n_nos_total"],
    res_val.loc[referencia, "n_nos_total"],
)

cond_ganho = abs(ganho_f1) < 0.05
cond_tempo = om_tempo >= 1.0
cond_nos = om_nos >= 1.0

print(f"modelo avaliado: {melhor_ensemble}  (referência: {referencia})")
print(f"  ganho de F1              {ganho_f1:+.4f}   < 0,05 ? {cond_ganho}")
print(f"  tempo, ordens de magn.   {om_tempo:+.2f}     >= 1  ? {cond_tempo}")
print(f"  nós, ordens de magnitude {om_nos:+.2f}     >= 1  ? {cond_nos}")
print()
print("H2 corroborada." if (cond_ganho and cond_tempo and cond_nos)
      else "H2 NÃO corroborada em todos os critérios.")

modelo avaliado: XGBoost  (referência: Árvore de Decisão)
  ganho de F1              +0.0012   < 0,05 ? True
  tempo, ordens de magn.   +0.90     >= 1  ? False
  nós, ordens de magnitude +0.20     >= 1  ? False

H2 NÃO corroborada em todos os critérios.


## 6. Congelamento dos resultados internos

A partir deste ponto, nenhum hiperparâmetro é alterado e nenhum modelo é
retreinado. O conjunto independente é consultado a seguir, uma única vez.

In [10]:
res_val.to_csv(TABELAS / "resultados_validacao.csv")

registro_h1_h2 = {
    "semente": SEMENTE,
    "n_treino": int(len(X_tr)),
    "n_validacao": int(len(X_val)),
    "n_atributos": int(X_tr.shape[1]),
    "f1_base_max": float(f1_base_max),
    "f1_ensemble_max": float(f1_ens_max),
    "melhor_ensemble": melhor_ensemble,
    "h1_corroborada": bool(f1_ens_max > f1_base_max),
    "h2_ganho_f1": float(ganho_f1),
    "h2_om_tempo": float(om_tempo),
    "h2_om_nos": float(om_nos),
    "h2_corroborada": bool(cond_ganho and cond_tempo and cond_nos),
}

with open(TABELAS / "hipoteses_h1_h2.json", "w", encoding="utf-8") as f:
    json.dump(registro_h1_h2, f, indent=2, ensure_ascii=False)

print(json.dumps(registro_h1_h2, indent=2, ensure_ascii=False))

{
  "semente": 42,
  "n_treino": 188296,
  "n_validacao": 47074,
  "n_atributos": 37,
  "f1_base_max": 0.9957192633150821,
  "f1_ensemble_max": 0.9969583644976315,
  "melhor_ensemble": "XGBoost",
  "h1_corroborada": true,
  "h2_ganho_f1": 0.0012391011825494402,
  "h2_om_tempo": 0.9009033728379132,
  "h2_om_nos": 0.19559358058790877,
  "h2_corroborada": false
}


## 7. H3 - generalização para conjunto independente

**Hipótese:** o F1 cai 0,10 ou mais em valor absoluto quando os modelos são
aplicados ao conjunto de Hannousse e Yahiouche.

Esta é a primeira e única consulta ao conjunto independente. Os modelos foram
treinados exclusivamente no PhiUSIIL e não recebem qualquer ajuste aqui.

In [11]:
registros_teste = [
    {"modelo": nome, **avaliar_predicao(modelo, X_teste, y_teste)}
    for nome, modelo in modelos.items()
]

res_teste = pd.DataFrame(registros_teste).set_index("modelo")
res_teste[["f1", "precisao", "recall", "roc_auc", "pr_auc"]].round(4)

,f1,precisao,recall,roc_auc,pr_auc
modelo,,,,,
Regressão Logística,0.6650,0.4989,0.9972,0.6547,0.6007
Árvore de Decisão,0.6651,0.4989,0.9977,0.5013,0.4989
Random Forest,0.6650,0.4988,0.9974,0.5010,0.4987
XGBoost,0.6655,0.4991,0.9984,0.6339,0.5808


In [12]:
res_teste[["f1", "precisao", "recall", "vp", "fp", "vn", "fn"]].round(4)

,f1,precisao,recall,vp,fp,vn,fn
modelo,,,,,,,
Regressão Logística,0.6650,0.4989,0.9972,5656,5682,31,16
Árvore de Decisão,0.6651,0.4989,0.9977,5659,5685,28,13
Random Forest,0.6650,0.4988,0.9974,5657,5684,29,15
XGBoost,0.6655,0.4991,0.9984,5663,5684,29,9


In [13]:
comparacao = pd.DataFrame({
    "f1_validacao": res_val["f1"],
    "f1_teste_independente": res_teste["f1"],
})
comparacao["queda_absoluta"] = (
    comparacao["f1_validacao"] - comparacao["f1_teste_independente"]
)
comparacao["h3_por_modelo"] = comparacao["queda_absoluta"] >= 0.10

comparacao.round(4)

,f1_validacao,f1_teste_independente,queda_absoluta,h3_por_modelo
modelo,,,,
Regressão Logística,0.9951,0.6650,0.3301,True
Árvore de Decisão,0.9957,0.6651,0.3306,True
Random Forest,0.9969,0.6650,0.3319,True
XGBoost,0.9970,0.6655,0.3315,True


In [14]:
queda_melhor = float(
    comparacao.loc[melhor_ensemble, "queda_absoluta"]
)
h3_corroborada = queda_melhor >= 0.10

print(f"modelo avaliado: {melhor_ensemble}")
print(f"  F1 na validação            {comparacao.loc[melhor_ensemble, 'f1_validacao']:.4f}")
print(f"  F1 no teste independente   {comparacao.loc[melhor_ensemble, 'f1_teste_independente']:.4f}")
print(f"  queda absoluta             {queda_melhor:.4f}   >= 0,10 ? {h3_corroborada}")
print()
print("H3 corroborada." if h3_corroborada else "H3 NÃO corroborada.")

modelo avaliado: XGBoost
  F1 na validação            0.9970
  F1 no teste independente   0.6655
  queda absoluta             0.3315   >= 0,10 ? True

H3 corroborada.


In [15]:
comparacao.to_csv(TABELAS / "comparacao_validacao_teste.csv")
res_teste.to_csv(TABELAS / "resultados_teste_independente.csv")

registro_h3 = {
    "n_teste_independente": int(len(X_teste)),
    "queda_f1_melhor_ensemble": queda_melhor,
    "h3_corroborada": bool(h3_corroborada),
    "quedas_por_modelo": comparacao["queda_absoluta"].round(4).to_dict(),
}

with open(TABELAS / "hipoteses_h3.json", "w", encoding="utf-8") as f:
    json.dump(registro_h3, f, indent=2, ensure_ascii=False)

print(json.dumps(registro_h3, indent=2, ensure_ascii=False))

{
  "n_teste_independente": 11385,
  "queda_f1_melhor_ensemble": 0.33146685500823736,
  "h3_corroborada": true,
  "quedas_por_modelo": {
    "Regressão Logística": 0.3301,
    "Árvore de Decisão": 0.3306,
    "Random Forest": 0.3319,
    "XGBoost": 0.3315
  }
}


## 8. Síntese

In [16]:
sintese = pd.DataFrame({
    "F1 validação": res_val["f1"],
    "F1 teste ind.": res_teste["f1"],
    "Queda": comparacao["queda_absoluta"],
    "Tempo (µs/URL)": res_val["tempo_mediana_us"],
    "Nós": res_val["n_nos_total"],
}).round(4)

sintese.to_csv(TABELAS / "sintese_final.csv")
sintese

,F1 validação,F1 teste ind.,Queda,Tempo (µs/URL),Nós
modelo,,,,,
Regressão Logística,0.9951,0.6650,0.3301,0.3384,37
Árvore de Decisão,0.9957,0.6651,0.3306,0.1990,2649
Random Forest,0.9969,0.6650,0.3319,10.1719,277014
XGBoost,0.9970,0.6655,0.3315,1.5841,4156


## Apêndice - exploração não reportada

> As células a seguir foram executadas **depois** do congelamento dos
> resultados internos (seção 6) e da consulta ao conjunto independente
> (H3, seção 7) e da síntese (seção 8). Nenhum resultado aqui altera,
> substitui ou é usado pelos resultados reportados nas seções 1-8: são
> investigações pontuais feitas a posteriori, mantidas por transparência
> em vez de apagadas.


### A. Diagnóstico de distribuição entre PhiUSIIL e Hannousse (pós-síntese)

> Investigação sobre diferenças estruturais de URL entre os dois
> datasets (contagem de duplicatas, exemplos de URLs, comparação de
> médias de atributos entre as classes legítimas de cada conjunto).


In [17]:
BRUTOS = Path("../dados/brutos")

phiusiil = pd.read_csv(BRUTOS / "PhiUSIIL_Phishing_URL_Dataset.csv")
hann = pd.read_csv(BRUTOS / "dataset_B_05_2020.csv")

phiusiil_limpo = phiusiil[~phiusiil["URL"].duplicated(keep="first")]
hann_limpo = hann[
    ~hann["url"].duplicated(keep="first")
    & ~hann["url"].isin(set(phiusiil_limpo["URL"]))
]

print(len(phiusiil_limpo), len(hann_limpo))  

235370 11385


In [18]:
print(hann_limpo[["url", "status"]].head(8).to_string())
print()
print(pd.crosstab(hann_limpo["status"].values, y_teste.values))

                                                                                                                              url      status
0                                                                                           http://www.crestonwood.com/router.php  legitimate
1                                                   http://shadetreetechnology.com/V4/validation/a111aedc8ae390eabcfa130e041a10a4    phishing
2  https://support-appleld.com.secureupdate.duilawyeryork.com/ap/89e6a3b4b063b8d/?cmd=_update&dispatch=89e6a3b4b063b8d1b&locale=_    phishing
3                                                                                                              http://rgipt.ac.in  legitimate
4                                                                         http://www.iracing.com/tracks/gateway-motorsports-park/  legitimate
5                                                                                                http://appleid.apple.com-app.es/    phishing
6     

In [19]:
leg_tr = phiusiil_limpo.loc[phiusiil_limpo["label"] == 1, "URL"].head(6).tolist()
for u in leg_tr:
    print("PhiUSIIL leg :", u[:80])

PhiUSIIL leg : https://www.southbankmosaics.com
PhiUSIIL leg : https://www.uni-mainz.de
PhiUSIIL leg : https://www.voicefmradio.co.uk
PhiUSIIL leg : https://www.sfnmjournal.com
PhiUSIIL leg : https://www.rewildingargentina.org
PhiUSIIL leg : https://www.globalreporting.org


In [20]:
comp = pd.DataFrame({
    "phiusiil_leg": X_val[y_val.values == 0].mean(),
    "hannousse_leg": X_teste[y_teste.values == 0].mean(),
})
print(comp.loc[["scheme_is_https", "path_length", "num_tokens_path",
                "url_length", "count_slash", "count_digits"]].round(3))

                 phiusiil_leg  hannousse_leg
scheme_is_https         1.000          0.445
path_length             0.000         20.168
num_tokens_path         0.000          1.368
url_length             27.207         47.391
count_slash             2.000          3.834
count_digits            0.050          1.529


In [21]:
res_val.to_csv(TABELAS / "resultados_validacao.csv")
res_teste.to_csv(TABELAS / "resultados_teste_independente.csv")
comp.to_csv(TABELAS / "comparacao_distribuicao_legitimas.csv")
sintese.to_csv(TABELAS / "sintese_final.csv")

### B. Investigação post-hoc: vazamento e poder discriminativo de atributos isolados

> Testes exploratórios feitos após ver o resultado de H3: um Random
> Forest sem 5 atributos julgados "espúrios", e Random Forests com um
> único atributo (path_length, url_length, shannon_entropy_hostname,
> count_digits).

In [22]:
espurios = ["path_length", "num_tokens_path", "count_slash",
            "scheme_is_https", "query_length"]
cols = [c for c in X_tr.columns if c not in espurios]

from sklearn.ensemble import RandomForestClassifier
rf2 = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                             random_state=42, n_jobs=1).fit(X_tr[cols], y_tr)

r_val = avaliar_predicao(rf2, X_val[cols], y_val)
r_te  = avaliar_predicao(rf2, X_teste[cols], y_teste)

print(f"validação  F1={r_val['f1']:.4f}  AUC={r_val['roc_auc']:.4f}")
print(f"teste ind. F1={r_te['f1']:.4f}  AUC={r_te['roc_auc']:.4f}  VN={r_te['vn']}")

registros_posthoc = [
    {"analise": "ablacao_sem_atributos_espurios", "atributo": None, "conjunto": "validacao", **r_val},
    {"analise": "ablacao_sem_atributos_espurios", "atributo": None, "conjunto": "teste_independente", **r_te},
]

validação  F1=0.9936  AUC=0.9984
teste ind. F1=0.6659  AUC=0.5867  VN=42


In [23]:
rf3 = RandomForestClassifier(n_estimators=50, class_weight="balanced",
                             random_state=42, n_jobs=1).fit(X_tr[["path_length"]], y_tr)
r_path_length = avaliar_predicao(rf3, X_val[["path_length"]], y_val)
print("F1 validação com UM atributo:", round(r_path_length["f1"], 4))

registros_posthoc.append({
    "analise": "atributo_unico", "atributo": "path_length", "conjunto": "validacao",
    **r_path_length,
})

F1 validação com UM atributo: 0.7491


In [24]:
for atr in ["url_length", "shannon_entropy_hostname", "count_digits"]:
    rf = RandomForestClassifier(n_estimators=50, class_weight="balanced",
                                random_state=42, n_jobs=1).fit(X_tr[[atr]], y_tr)
    r = avaliar_predicao(rf, X_val[[atr]], y_val)
    print(f"{atr:26s} F1 validação = {r['f1']:.4f}")
    registros_posthoc.append({
        "analise": "atributo_unico", "atributo": atr, "conjunto": "validacao",
        **r,
    })

url_length                 F1 validação = 0.6379


shannon_entropy_hostname   F1 validação = 0.6313


count_digits               F1 validação = 0.6238


In [25]:
analises_posthoc = pd.DataFrame(registros_posthoc)
analises_posthoc.to_csv(TABELAS / "analises_posthoc.csv", index=False)
analises_posthoc

,analise,atributo,conjunto,f1,precisao,recall,roc_auc,pr_auc,vp,fp,vn,fn
0,ablacao_sem_atributos_espurios,NaN,validacao,0.993558,0.993681,0.993434,0.998432,0.998399,19972,127,26843,132
1,ablacao_sem_atributos_espurios,NaN,teste_independente,0.665922,0.499603,0.998237,0.586674,0.553294,5662,5671,42,10
2,atributo_unico,path_length,validacao,0.749090,1.000000,0.598836,0.799418,0.770162,12039,0,26970,8065
3,atributo_unico,url_length,validacao,0.637907,0.795276,0.532531,0.752769,0.766207,10706,2756,24214,9398
4,atributo_unico,shannon_entropy_hostname,validacao,0.631332,0.783844,0.528502,0.759343,0.770084,10625,2930,24040,9479
5,atributo_unico,count_digits,validacao,0.623797,0.931075,0.469011,0.725380,0.686776,9429,698,26272,10675


### C. Intervalos de confiança do F1 (bootstrap)

In [26]:
from avaliacao import ic_bootstrap_f1

registros_ic = []
for nome, modelo in modelos.items():
    ic = ic_bootstrap_f1(modelo, X_val, y_val, n_reamostras=1000, semente=SEMENTE)
    registros_ic.append({"modelo": nome, **ic})
    print(f"{nome:22s} F1={ic['f1_observado']:.4f}  "
          f"IC95=[{ic['ic95_inferior']:.4f}, {ic['ic95_superior']:.4f}]")

intervalos_confianca_f1 = pd.DataFrame(registros_ic).set_index("modelo")
intervalos_confianca_f1.to_csv(TABELAS / "intervalos_confianca_f1.csv")
intervalos_confianca_f1

Regressão Logística    F1=0.9951  IC95=[0.9945, 0.9958]


Árvore de Decisão      F1=0.9957  IC95=[0.9951, 0.9964]


Random Forest          F1=0.9969  IC95=[0.9963, 0.9974]


XGBoost                F1=0.9970  IC95=[0.9964, 0.9975]


,f1_observado,ic95_inferior,ic95_superior,n_reamostras
modelo,,,,
Regressão Logística,0.995132,0.994472,0.995819,1000
Árvore de Decisão,0.995719,0.995062,0.996357,1000
Random Forest,0.996910,0.996337,0.997440,1000
XGBoost,0.996958,0.996378,0.997492,1000


### D. Comparação custo-benefício entre modelos

In [27]:
recomendacao = pd.DataFrame({
    "F1 validação":        res_val["f1"],
    "F1 teste ind.":       res_teste["f1"],
    "ROC-AUC teste ind.":  res_teste["roc_auc"],
    "VN teste ind.":       res_teste["vn"],
    "Tempo (µs/URL)":      res_val["tempo_mediana_us"],
    "Complexidade":        res_val["n_nos_total"],
    "Tipo complexidade":   res_val["tipo_complexidade"],
})

recomendacao.to_csv(TABELAS / "comparacao_custo_beneficio.csv")
recomendacao.round(4)

,F1 validação,F1 teste ind.,ROC-AUC teste ind.,VN teste ind.,Tempo (µs/URL),Complexidade,Tipo complexidade
modelo,,,,,,,
Regressão Logística,0.9951,0.6650,0.6547,31,0.3384,37,coeficientes
Árvore de Decisão,0.9957,0.6651,0.5013,28,0.1990,2649,nos_arvore
Random Forest,0.9969,0.6650,0.5010,29,10.1719,277014,nos_arvore
XGBoost,0.9970,0.6655,0.6339,29,1.5841,4156,nos_arvore
